# GROMACS MD Simulation on Free GPU

**Goal:** Complete 10 ns MD simulation with free T4 GPU  
**Time:** ~8-12 hours (vs 5 days on CPU)  
**Cost:** FREE

---

## ⚠️ BEFORE YOU START

### 1. Enable GPU Runtime
- **Runtime** → **Change runtime type**
- **Hardware accelerator** → **GPU** (T4)
- Click **Save**

### 2. Upload File to Google Drive
- Upload `colab_fresh_start.zip` (4.2 MB) to your Google Drive **root folder**
- Path should be: `MyDrive/colab_fresh_start.zip`

---


## Step 1: Verify GPU

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  WARNING: No GPU detected!")
    print("   Go to Runtime → Change runtime type → GPU → Save")

## Step 2: Install GROMACS with GPU Support

**Time:** ~15-20 minutes  
**Note:** We use conda because apt's GROMACS doesn't have GPU support

In [ ]:
%%time
import os

# Download and install Miniconda
print("📦 Installing Miniconda...")
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O miniconda.sh
!bash miniconda.sh -b -p /content/miniconda > /dev/null 2>&1
!rm miniconda.sh

# Add conda to PATH
os.environ['PATH'] = f"/content/miniconda/bin:{os.environ['PATH']}"

# Install GROMACS with GPU support from conda-forge
print("\n🧬 Installing GROMACS with GPU support...")
!/content/miniconda/bin/conda install -c conda-forge gromacs -y -q

# Verify installation
print("\n📋 GROMACS Version:")
!/content/miniconda/bin/gmx --version | grep -E "GROMACS version|GPU support"

print("\n✅ GROMACS installed successfully!")

## Step 3: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify file exists
import os
zip_path = '/content/drive/MyDrive/colab_fresh_start.zip'
if os.path.exists(zip_path):
    print(f"\n✅ Found: {zip_path}")
    !ls -lh {zip_path}
else:
    print(f"\n❌ ERROR: File not found at {zip_path}")
    print("   Please upload 'colab_fresh_start.zip' to Google Drive root folder")
    print("   Then re-run this cell")

## Step 4: Extract System Files

In [ ]:
# Extract files to working directory
!cd /content && unzip -o /content/drive/MyDrive/colab_fresh_start.zip

# Verify files
print("\n📁 Extracted files:")
!ls -lh /content/system.gro /content/topol.top /content/md.mdp

print("\n✅ Files ready!")

## Step 5: Prepare MD Input

**Time:** ~1-2 minutes

In [ ]:
%%time
# Generate binary input file for mdrun
!/content/miniconda/bin/gmx grompp \
  -f /content/md.mdp \
  -c /content/system.gro \
  -p /content/topol.top \
  -o /content/md.tpr \
  -maxwarn 2

# Verify output
print("\n📋 MD input file:")
!ls -lh /content/md.tpr

print("\n✅ Ready to run MD simulation!")

## Step 6: Run MD Simulation 🚀

**⏰ Time:** ~8-12 hours for 10 ns  
**⚠️ Important:** This will run for hours. You can close the browser (minimize tab), but don't close the tab completely.

**Progress updates every 10 ps:**
```
Step 50000, time 100.0 ps
Performance: 28.5 ns/day
```

**💡 Tip:** Click the three dots (...) on the cell → "Show/Hide output" to collapse output after it starts running

In [ ]:
%%time
# Run MD with GPU acceleration
!/content/miniconda/bin/gmx mdrun \
  -v \
  -deffnm /content/md \
  -ntomp 2 \
  -nb gpu \
  -pme gpu \
  -bonded gpu \
  -update gpu

# Flags:
# -v             : verbose (show progress)
# -deffnm md     : default filename prefix
# -ntomp 2       : use 2 CPU threads (Colab has 2 vCPUs)
# -nb gpu        : non-bonded interactions on GPU
# -pme gpu       : particle mesh Ewald electrostatics on GPU
# -bonded gpu    : bonded interactions on GPU
# -update gpu    : coordinate updates on GPU

print("\n🎉 Simulation complete!")

## Step 7: Check Results

In [ ]:
# Check output files
print("📁 Output files:")
!ls -lh /content/md.xtc /content/md.log /content/md.edr /content/md.cpt

# Check final progress
print("\n📊 Final progress:")
!tail -100 /content/md.log | grep "Step" | tail -3

# Check performance
print("\n⚡ Performance:")
!grep "Performance" /content/md.log | tail -5

# Verify completion
!grep "Finished mdrun" /content/md.log && echo "\n✅ Simulation completed successfully!" || echo "\n⚠️ Simulation may not be complete"

## Step 8: Save Results to Google Drive

**Time:** ~5-10 minutes (compressing ~500-1000 MB)

In [ ]:
%%time
# Compress results
print("📦 Compressing results...")
!cd /content && zip -q md_results.zip md.xtc md.log md.edr md.cpt

# Check size
print("\n📋 Compressed file:")
!ls -lh /content/md_results.zip

# Copy to Google Drive
print("\n☁️  Uploading to Google Drive...")
!cp /content/md_results.zip /content/drive/MyDrive/

print("\n✅ Results saved to Google Drive!")
print("   Download 'md_results.zip' from: https://drive.google.com")
print("   File size: ~500-1000 MB")

---

## 🔄 Emergency: Save Checkpoint (If Session Will Timeout)

**Only run this if:**
- You need to stop before Step 6 completes
- Session is about to timeout (11+ hours)
- You want to resume later

**How to use:**
1. Stop Step 6 cell (click stop button)
2. Wait 30 seconds for checkpoint to save
3. Run this cell

In [ ]:
# Save checkpoint and trajectory so far
print("💾 Saving checkpoint...")
!cp /content/md.cpt /content/drive/MyDrive/md_checkpoint.cpt
!cp /content/md.tpr /content/drive/MyDrive/md.tpr
!cp /content/topol.top /content/drive/MyDrive/topol.top
!cp /content/md.log /content/drive/MyDrive/md_partial.log
!cp /content/md.xtc /content/drive/MyDrive/md_partial.xtc 2>/dev/null || echo "No trajectory yet"

# Show progress
print("\n📊 Saved at:")
!tail -5 /content/md.log | grep "Step" || echo "Check log file"

print("\n✅ Checkpoint saved to Google Drive!")
print("   Safe to disconnect now.")
print("   To resume: run 'Resume Simulation' cell below")

## 🔄 Resume Simulation (After Reconnect)

**When to use:**
- Session timed out (>12 hours)
- You stopped and saved checkpoint
- Want to continue from where you left off

**How to use:**
1. Re-run Cells 1-2 (verify GPU, install GROMACS)
2. Re-run Cell 3 (mount Drive)
3. Run this cell to restore and resume

In [ ]:
%%time
# Restore checkpoint from Drive
print("📥 Restoring checkpoint...")
!cp /content/drive/MyDrive/md_checkpoint.cpt /content/md.cpt
!cp /content/drive/MyDrive/md.tpr /content/md.tpr
!cp /content/drive/MyDrive/topol.top /content/topol.top

# Verify files
!ls -lh /content/md.cpt /content/md.tpr

# Resume simulation
print("\n🔄 Resuming simulation...")
!/content/miniconda/bin/gmx mdrun \
  -v \
  -deffnm /content/md \
  -cpi /content/md.cpt \
  -ntomp 2 \
  -nb gpu \
  -pme gpu \
  -bonded gpu \
  -update gpu

print("\n✅ Resumed simulation complete!")
print("   Now run Step 7 (Check Results) and Step 8 (Save to Drive)")

---

## 📊 Expected Timeline

| Step | Time | What Happens |
|------|------|-------------|
| 1. GPU Check | 10 sec | Verify T4 GPU available |
| 2. Install GROMACS | 15-20 min | Download conda + GROMACS |
| 3. Mount Drive | 30 sec | Authorize Google Drive |
| 4. Extract Files | 10 sec | Unzip system files |
| 5. Prepare Input | 1-2 min | Generate .tpr file |
| 6. **MD Simulation** | **8-12 hours** | **Main simulation** |
| 7. Check Results | 10 sec | Verify completion |
| 8. Save to Drive | 5-10 min | Compress & upload |
| **TOTAL** | **~9-13 hours** | **vs 5 days on CPU!** |

---

## 🐛 Troubleshooting

### "No GPU detected"
→ Runtime → Change runtime type → GPU → Save → Restart runtime

### "File not found: colab_fresh_start.zip"
→ Upload to Google Drive **root folder** (not in any subfolder)  
→ Path should be: `MyDrive/colab_fresh_start.zip`

### "gmx: command not found"
→ Re-run Step 2 (Install GROMACS)

### "GPU support disabled"
→ Make sure you're using conda GROMACS (Step 2)  
→ Don't install via apt-get

### "Session disconnected" (after 12 hours)
→ Run checkpoint save cell before timeout  
→ Restart session and use resume cell  
→ Or upgrade to Colab Pro ($10/month, 24-hour sessions)

### Simulation crashes
→ Check: `!tail -100 /content/md.log`  
→ Look for "Fatal error" messages

---

## ✅ Success Criteria

**Simulation is complete when:**
- ✅ Step reaches 5,000,000
- ✅ Time reaches 10,000 ps (10 ns)
- ✅ Log shows "Finished mdrun"
- ✅ md.xtc file is ~500-1000 MB
- ✅ Performance ~20-30 ns/day on T4 GPU

---

## 💡 Pro Tips

1. **Keep browser open** - Minimize tab, but don't close completely
2. **Save checkpoints** - Every few hours if simulation is long
3. **Colab Pro** - $10/month for 24-hour sessions (no interruption)
4. **Download immediately** - After Step 8, download from Drive right away
5. **Delete from Colab** - Free up space after downloading

---

*Created: January 27, 2026*  
*Project: p53 StabiliMut Initiative 2*  
*Version: Final (GPU-enabled, checkpoint-ready)*
